# Does a coarser camera hide small ships?

Ship detection on Airbus SPOT satellite tiles at 1.5 m ground resolution, then on the same tiles degraded to look like a 3 m and a 6 m camera.

**Question.** Small satellites carry small cameras, which see the sea far less sharply than large imaging satellites. How much does ship detection suffer when the same scene is seen at a coarser ground resolution?

**Method.** Train a small YOLO11n detector on 1.5 m tiles. Score it on held-out tiles at 1.5 m, then on copies of the same tiles degraded to 3 m and 6 m: blur for the optics, block-averaging for bigger ground pixels, JPEG for on-board compression, then resampling back to 768 × 768 so ships keep their size on screen and only detail is lost.

**Outputs.** `results.csv` with mAP50, recall and precision at each resolution, and `one_picture.png`, one tile at each resolution with real ships in red and detections in green.

**Running it on Kaggle.**
1. Accept the rules of the [Airbus Ship Detection Challenge](https://www.kaggle.com/c/airbus-ship-detection) and add its data as input.
2. In the notebook settings, choose **GPU T4 x2** and turn **Internet on**.
3. Run the cells in order, one at a time. Training takes roughly 20–45 minutes on a T4 (estimate).

If a cell fails, read the last lines of the error: most problems come from the data not being attached, or the GPU or internet setting being off.

## 1 · Settings

Tile counts, training length and the coarser cameras to imitate. These are the only numbers meant to be changed.

**Expected output:** `Settings loaded.`

In [ ]:
import os

N_SHIP_TILES = 4000      # tiles WITH ships to use (more = slower, a little better)
EMPTY_SHARE = 0.20       # share of empty sea in the set, so the model also learns "no ship"
VAL_SHARE = 0.15         # share of tiles kept aside for testing, never used in training
EPOCHS = 15              # passes over all training tiles
IMG_SIZE = 768           # Airbus tiles are 768 x 768 pixels: keep full size so small ships stay visible
SHRINK_FACTORS = [2, 4]  # 2 turns the 1.5 m tiles into a 3 m camera, 4 into a 6 m camera
JPEG_QUALITY = 85        # small satellites compress images before downlink; 85 is a common setting
SEED = 0                 # fixes the random choices, so a re-run picks the same tiles

INPUT = os.environ.get("AIRBUS_INPUT", "/kaggle/input")    # Kaggle's data folder (the variable is only for local testing)
WORK = os.environ.get("NB_WORK", "/kaggle/working")        # where everything this notebook makes is saved
print("Settings loaded.")

## 2 · Install the detector library and check the GPU

Installs Ultralytics (the YOLO library, at the version this notebook was tested with), switches off outside logging services so nothing asks for a login, and checks for a GPU.

**Expected output:** `GPU: Tesla T4`. If it says *none found*, turn on **GPU T4 x2** in the settings and run cells 1 and 2 again. About 1–2 minutes.

In [ ]:
!pip install -q ultralytics==8.4.216

import random, shutil, time
from pathlib import Path

os.environ["WANDB_MODE"] = "disabled"   # stops training from asking for a Weights & Biases login

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO, settings

for service in ["wandb", "mlflow", "comet", "clearml", "dvc", "neptune", "tensorboard", "raytune", "hub", "sync"]:
    if service in settings:
        settings.update({service: False})   # no outside logging services

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none found")
random.seed(SEED)
np.random.seed(SEED)

## 3 · Find the Airbus data

Finds the competition files: a folder of 768 × 768 tiles and a table listing every ship as a mask.

**Expected output:** the data folder, then three counts: tiles, tiles with at least one ship, and ships. Most tiles are empty sea.

In [ ]:
inputs = Path(INPUT)
candidates = [inputs / "airbus-ship-detection"] + sorted(inputs.glob("*")) + sorted(inputs.glob("*/*"))
DATA_DIR = next((p for p in candidates if (p / "train_ship_segmentations_v2.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Airbus data not found. Add Input > Competitions > 'Airbus Ship Detection Challenge'.")
IMG_DIR = DATA_DIR / "train_v2"
masks = pd.read_csv(DATA_DIR / "train_ship_segmentations_v2.csv")
has_ship = masks["EncodedPixels"].notna()
print("Data folder:", DATA_DIR)
print(f"Tiles: {masks['ImageId'].nunique():,} | tiles with ships: {masks.loc[has_ship, 'ImageId'].nunique():,} | ships: {has_ship.sum():,}")

## 4 · Turn ship masks into boxes (labels)

Airbus stores each ship's shape as runs of pixels counted down each column (run-length encoding). A detector learns from boxes, so each shape becomes the smallest upright box around it. The cell then picks 4,000 tiles with ships and 1,000 of empty sea, keeps 15% aside as test (validation) tiles, copies everything into the folder layout YOLO expects, and writes one label file per tile.

**Expected output:** progress every 1,000 tiles, then `Tiles copied: 5000 (train 4250, val 750)` and a summary of ship lengths in pixels. About 3–6 minutes.

At 1.5 m per pixel, a 30 m fishing boat is about 20 pixels long; the 10% line of the summary shows the small ships that suffer most at 6 m.

In [ ]:
SIZE = 768   # every Airbus tile is 768 x 768 pixels
GSD = 1.5    # SPOT ground resolution: 1.5 metres per pixel


def rle_to_mask(rle, size=SIZE):
    """Airbus lists ship pixels as 'start length' pairs, counting down each column, then across."""
    nums = np.asarray(rle.split(), dtype=np.int64)
    starts, lengths = nums[0::2] - 1, nums[1::2]
    flat = np.zeros(size * size, dtype=np.uint8)
    for s, n in zip(starts, lengths):
        flat[s:s + n] = 1
    return flat.reshape(size, size).T   # counted by columns, so turn it into rows x columns


def mask_to_box(mask):
    """Smallest upright box around a ship: left, top, right, bottom, in pixels."""
    ys, xs = np.nonzero(mask)
    return xs.min(), ys.min(), xs.max() + 1, ys.max() + 1


rng = random.Random(SEED)
ship_ids = sorted(masks.loc[has_ship, "ImageId"].unique())
empty_ids = sorted(masks.loc[~has_ship, "ImageId"].unique())
chosen_ships = rng.sample(ship_ids, min(N_SHIP_TILES, len(ship_ids)))
n_empty = int(round(len(chosen_ships) * EMPTY_SHARE / (1 - EMPTY_SHARE)))
tiles = chosen_ships + rng.sample(empty_ids, min(n_empty, len(empty_ids)))
rng.shuffle(tiles)
n_val = int(round(len(tiles) * VAL_SHARE))
split = {t: ("val" if i < n_val else "train") for i, t in enumerate(tiles)}

DATASET = Path(WORK) / "ships"
for part in ["train", "val"]:
    (DATASET / "images" / part).mkdir(parents=True, exist_ok=True)
    (DATASET / "labels" / part).mkdir(parents=True, exist_ok=True)

rles = masks[has_ship].groupby("ImageId")["EncodedPixels"].apply(list).to_dict()
stats = []
for i, tile in enumerate(tiles, 1):
    part = split[tile]
    lines = []
    for rle in rles.get(tile, []):
        x0, y0, x1, y1 = mask_to_box(rle_to_mask(rle))
        xc, yc, w, h = (x0 + x1) / 2 / SIZE, (y0 + y1) / 2 / SIZE, (x1 - x0) / SIZE, (y1 - y0) / SIZE
        lines.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")   # class 0 = ship, then centre and size as fractions of the tile
        stats.append({"tile": tile, "split": part, "length_px": int(max(x1 - x0, y1 - y0))})
    (DATASET / "labels" / part / (Path(tile).stem + ".txt")).write_text("\n".join(lines))
    shutil.copy(IMG_DIR / tile, DATASET / "images" / part / tile)
    if i % 1000 == 0:
        print(f"  {i} tiles done")

DATA_YAML = DATASET / "ships.yaml"
DATA_YAML.write_text(f"path: {DATASET}\ntrain: images/train\nval: images/val\nnames:\n  0: ship\n")
ship_stats = pd.DataFrame(stats)
print(f"Tiles copied: {len(tiles)} (train {len(tiles) - n_val}, val {n_val})")
print("Ship length in pixels (longest side of its box):")
print(ship_stats["length_px"].describe(percentiles=[0.1, 0.5, 0.9]).round(1).to_string())

## 5 · Check the labels by eye

Draws six random training tiles with their boxes in red.

**Expected output:** every red box sitting on a ship, and no ship without a box.

In [ ]:
def read_boxes(label_path, size=SIZE):
    """Read a label file back into pixel boxes: left, top, right, bottom."""
    boxes = []
    for line in Path(label_path).read_text().strip().splitlines():
        _, xc, yc, w, h = map(float, line.split())
        boxes.append(((xc - w / 2) * size, (yc - h / 2) * size, (xc + w / 2) * size, (yc + h / 2) * size))
    return boxes


train_with_ships = [t for t in tiles if split[t] == "train" and t in rles]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, tile in zip(axes.flat, random.Random(SEED).sample(train_with_ships, min(6, len(train_with_ships)))):
    img = cv2.cvtColor(cv2.imread(str(DATASET / "images" / "train" / tile)), cv2.COLOR_BGR2RGB)
    for x0, y0, x1, y1 in read_boxes(DATASET / "labels" / "train" / (Path(tile).stem + ".txt")):
        cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), (255, 0, 0), 2)
    ax.imshow(img)
    ax.set_title(tile)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6 · Train the detector

Worth doing first: write down a prediction of mAP50 (0 to 1) at 1.5 m, 3 m and 6 m.

Starts from YOLO11n, a small detector pre-trained on everyday photos, and trains it on the ship tiles for 15 epochs. In each epoch it sees every training tile once, compares its boxes with the labels and adjusts.

**Expected output:** one block of numbers per epoch with `box_loss` mostly falling, then mAP50 on the test tiles and a plot of the training curves. Roughly 20–45 minutes on a T4 (estimate): time epoch 1 and multiply by 15.

In [ ]:
from IPython.display import Image as ShowImage, display

t0 = time.time()
model = YOLO("yolo11n.pt")
model.train(data=str(DATA_YAML), imgsz=IMG_SIZE, epochs=EPOCHS, batch=16, device=DEVICE, workers=4, seed=SEED,
            project=f"{WORK}/runs", name="ships", exist_ok=True, plots=True)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
print(f"Training took {(time.time() - t0) / 60:.0f} minutes. Best model saved at {BEST}")
display(ShowImage(filename=str(RUN_DIR / "results.png"), width=900))

## 7 · Make the 3 m and 6 m copies of the test tiles

A coarser camera blurs, because its optics spread each point of light, and samples the ground in bigger pixels. For each test tile:
1. **blur** slightly, a stand-in for the optics (the amount is an assumption, not a measured camera);
2. **shrink** by 2 or 4 by averaging blocks of pixels: bigger ground pixels;
3. **compress** as JPEG, as a small satellite does before downlink;
4. **enlarge** back to 768 × 768, so ships keep their size on screen and only detail is lost.

Labels are copied unchanged, because boxes are stored as fractions of the tile.

**Expected output:** `3 m copy made` and `6 m copy made`.

In [ ]:
def degrade(img, k, quality=JPEG_QUALITY):
    """Make a 1.5 m tile look like a camera k times coarser."""
    h, w = img.shape[:2]
    blurred = cv2.GaussianBlur(img, (0, 0), sigmaX=0.5 * k)                       # 1. optics blur (an assumption)
    small = cv2.resize(blurred, (w // k, h // k), interpolation=cv2.INTER_AREA)    # 2. bigger ground pixels
    ok, buf = cv2.imencode(".jpg", small, [cv2.IMWRITE_JPEG_QUALITY, quality])     # 3. on-board compression
    small = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)               # 4. back to 768 x 768


val_tiles = [t for t in tiles if split[t] == "val"]


def make_coarser_copy(k):
    """Save every test tile as a camera k times coarser would see it; return the copy's data file."""
    out = Path(WORK) / f"ships_{GSD * k:g}m"
    (out / "images" / "val").mkdir(parents=True, exist_ok=True)
    (out / "labels" / "val").mkdir(parents=True, exist_ok=True)
    for tile in val_tiles:
        img = cv2.imread(str(DATASET / "images" / "val" / tile))
        if img is None:
            continue   # skip any tile that can't be read
        cv2.imwrite(str(out / "images" / "val" / (Path(tile).stem + ".png")), degrade(img, k))
        shutil.copy(DATASET / "labels" / "val" / (Path(tile).stem + ".txt"), out / "labels" / "val")
    yaml_path = out / "ships.yaml"
    yaml_path.write_text(f"path: {out}\ntrain: images/val\nval: images/val\nnames:\n  0: ship\n")
    return yaml_path


YAMLS = {"1.5 m": DATA_YAML}
for k in SHRINK_FACTORS:
    YAMLS[f"{GSD * k:g} m"] = make_coarser_copy(k)
    print(f"{GSD * k:g} m copy made")

## 8 · See what each camera sees

One test tile at 1.5 m, 3 m and 6 m, zoomed in on a ship: the ship turns into a few blurry blocks as the camera gets coarser.

In [ ]:
val_with_ships = [t for t in val_tiles if t in rles]


def coarser_paths(tile):
    """The same tile at 1.5 m, then at each coarser camera."""
    return [DATASET / "images" / "val" / tile] + [Path(WORK) / f"ships_{GSD * k:g}m" / "images" / "val" / (Path(tile).stem + ".png") for k in SHRINK_FACTORS]


example = random.Random(SEED).choice(val_with_ships)
x0, y0, x1, y1 = read_boxes(DATASET / "labels" / "val" / (Path(example).stem + ".txt"))[0]
half = 96   # zoom window: 192 x 192 pixels around the first ship
xa = int(max(0, min(SIZE - 2 * half, (x0 + x1) / 2 - half)))
ya = int(max(0, min(SIZE - 2 * half, (y0 + y1) / 2 - half)))
fig, axes = plt.subplots(1, len(YAMLS), figsize=(15, 5))
for ax, res, path in zip(axes, YAMLS, coarser_paths(example)):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img[ya:ya + 2 * half, xa:xa + 2 * half], interpolation="nearest")
    ax.set_title(f"{res}, zoomed on one ship")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9 · Score the detector at all three resolutions

Runs the trained detector on the test tiles at each resolution and compares its boxes with the real ones.
- **mAP50:** how well it finds ships with boxes overlapping the real box by at least half, summarised over all confidence levels; 1 is perfect.
- **Recall:** the share of real ships found. **Precision:** the share of its boxes that really are ships.

Compare rows on mAP50: Ultralytics reports recall and precision at each run's own best-F1 confidence, so those two can shift between rows for reasons other than the camera.

**Expected output:** a three-row table, also saved as `results.csv`.

In [ ]:
best_model = YOLO(str(BEST))


def score(yaml_path, name):
    """Test the trained detector on one set of tiles; return its three scores."""
    m = best_model.val(data=str(yaml_path), imgsz=IMG_SIZE, batch=16, device=DEVICE, plots=False, verbose=False,
                       project=f"{WORK}/runs", name=f"val_{name}", exist_ok=True)
    return {"mAP50": round(float(m.box.map50), 3), "recall": round(float(m.box.mr), 3), "precision": round(float(m.box.mp), 3)}


results = pd.DataFrame([{"resolution": res, **score(path, res.replace(" ", ""))} for res, path in YAMLS.items()])
results.to_csv(f"{WORK}/results.csv", index=False)
print(results.to_string(index=False))

## 10 · One picture of the result

One test tile with several ships, drawn at each resolution: real ships in **red**, the detector's boxes in **green**, with how many ships it found and the mAP50 for that resolution. A red box with no green on it is a missed ship. "Found" means a green box overlaps the red one by at least half, at confidence 0.25.

**Expected output:** three panels, saved as `one_picture.png`.

In [ ]:
def overlap(a, b):
    """Shared area of two boxes divided by their combined area: 0 = no overlap, 1 = identical."""
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0


counts = {t: len(read_boxes(DATASET / "labels" / "val" / (Path(t).stem + ".txt"))) for t in val_with_ships}
busy = [t for t in val_with_ships if 3 <= counts[t] <= 10] or val_with_ships
tile = max(busy, key=lambda t: counts[t])
truth = read_boxes(DATASET / "labels" / "val" / (Path(tile).stem + ".txt"))

fig, axes = plt.subplots(1, len(YAMLS), figsize=(18, 6.5))
for ax, res, path in zip(axes, YAMLS, coarser_paths(tile)):
    img = cv2.imread(str(path))
    found_boxes = [b.tolist() for b in best_model.predict(img, imgsz=IMG_SIZE, conf=0.25, device=DEVICE, verbose=False)[0].boxes.xyxy.cpu()]
    found = sum(any(overlap(t, f) >= 0.5 for f in found_boxes) for t in truth)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for x0, y0, x1, y1 in truth:
        cv2.rectangle(rgb, (int(x0), int(y0)), (int(x1), int(y1)), (230, 40, 40), 3)
    for x0, y0, x1, y1 in found_boxes:
        cv2.rectangle(rgb, (int(x0), int(y0)), (int(x1), int(y1)), (40, 200, 70), 2)
    ax.imshow(rgb)
    ax.set_title(f"{res}: found {found} of {len(truth)} ships, mAP50 {results.loc[results['resolution'] == res, 'mAP50'].item():.2f}", fontsize=13)
    ax.axis("off")
fig.suptitle("Same tile, coarser cameras. Red: real ships. Green: detector's boxes.", fontsize=15)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.savefig(f"{WORK}/one_picture.png", dpi=130, bbox_inches="tight")
plt.show()

## 11 · Try another camera

Change the number after `K =` and run the cell: 8 gives a 12 m camera, 3 gives 4.5 m.

**Expected output:** one line with that camera's scores.

In [ ]:
K = 8   # shrink factor: 8 turns the 1.5 m tiles into a 12 m camera

print(f"{GSD * K:g} m camera:", score(make_coarser_copy(K), f"k{K}"))

## Results

| Resolution | mAP50 | Recall | Precision |
|---|---|---|---|
| 1.5 m | | | |
| 3 m | | | |
| 6 m | | | |

## Limitations
1. Airbus tiles are crops of larger scenes and some overlap, so a test tile can share ships with a training tile; the scores run high.
2. The blur and compression settings are assumptions, not a measured camera.
3. Timings come from a data-centre GPU (T4), not an embedded processor.
4. Upright boxes fit long, thin, rotated ships loosely.

## Next steps
- Retrain on degraded tiles and measure how much of the score comes back.
- Rotated (oriented) boxes for long, thin ships.
- Accuracy and speed on an embedded GPU, including INT8.